# Qwen3-4B Curriculum Learning Evaluation

This notebook evaluates all 5 stages (S1-S5) of the curriculum training on Colab T4 GPU.

**Expected Time**: ~3 minutes for all 5 stages (500 samples each)

**Prerequisites**: Upload `curriculum_models.zip` to Google Drive

## Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2: Extract Models and Data

In [ ]:
import os

# Create directories
os.makedirs('/content/ecommerce-review-analysis/models/curriculum', exist_ok=True)
os.makedirs('/content/ecommerce-review-analysis/data/curriculum', exist_ok=True)
os.makedirs('/content/ecommerce-review-analysis/code', exist_ok=True)

# Extract uploaded zip file
!unzip -q /content/drive/MyDrive/curriculum_models.zip -d /content/ecommerce-review-analysis/

# Verify extraction
!ls -lh /content/ecommerce-review-analysis/models/curriculum/

## Step 3: Clone Repository and Install Dependencies

In [ ]:
%cd /content/ecommerce-review-analysis/code
!git clone https://github.com/kotoriee/ecommerce-review-analysis.git .

# Install dependencies (Unsloth only, no vLLM needed)
!pip install unsloth trl datasets -q

## Step 4: Evaluate All Stages

In [ ]:
import sys
sys.path.insert(0, '/content/ecommerce-review-analysis/code')

### Evaluate S1 (600 samples training)

In [ ]:
!python colab/curriculum_eval_colab.py \
    --stage s1 \
    --model-dir /content/ecommerce-review-analysis/models/curriculum \
    --data-dir /content/ecommerce-review-analysis/data/curriculum \
    --samples 500

### Evaluate S2 (1200 samples training)

In [ ]:
!python colab/curriculum_eval_colab.py \
    --stage s2 \
    --model-dir /content/ecommerce-review-analysis/models/curriculum \
    --data-dir /content/ecommerce-review-analysis/data/curriculum \
    --samples 500

### Evaluate S3 (2400 samples training)

In [ ]:
!python colab/curriculum_eval_colab.py \
    --stage s3 \
    --model-dir /content/ecommerce-review-analysis/models/curriculum \
    --data-dir /content/ecommerce-review-analysis/data/curriculum \
    --samples 500

### Evaluate S4 (4800 samples training)

In [ ]:
!python colab/curriculum_eval_colab.py \
    --stage s4 \
    --model-dir /content/ecommerce-review-analysis/models/curriculum \
    --data-dir /content/ecommerce-review-analysis/data/curriculum \
    --samples 500

### Evaluate S5 (8500 samples training)

In [ ]:
!python colab/curriculum_eval_colab.py \
    --stage s5 \
    --model-dir /content/ecommerce-review-analysis/models/curriculum \
    --data-dir /content/ecommerce-review-analysis/data/curriculum \
    --samples 500

## Step 5: Summary and Visualization

In [ ]:
import json
import glob
import matplotlib.pyplot as plt

# Collect results
results = {}
for stage in ["s1", "s2", "s3", "s4", "s5"]:
    result_file = f"eval_{stage}_colab.json"
    matching = glob.glob(result_file)
    if matching:
        with open(matching[0]) as f:
            data = json.load(f)
        results[stage] = data

# Print summary
print("="*60)
print("Curriculum Learning Results Summary")
print("="*60)

stage_names = {
    "s1": "S1 (600)",
    "s2": "S2 (1200)",
    "s3": "S3 (2400)",
    "s4": "S4 (4800)",
    "s5": "S5 (8500)"
}

for stage in ["s1", "s2", "s3", "s4", "s5"]:
    if stage in results:
        data = results[stage]
        print(f"{stage_names[stage]}: {data['accuracy']:.2f}% "
              f"(correct: {data['correct']}/{data['total']}, "
              f"speed: {data['speed']:.1f} samples/sec)")

# Plot learning curve
if results:
    stages = ["S1(600)", "S2(1200)", "S3(2400)", "S4(4800)", "S5(8500)"]
    accuracies = [results.get(s, {}).get("accuracy", 0) for s in ["s1", "s2", "s3", "s4", "s5"]]
    
    plt.figure(figsize=(10, 6))
    plt.plot(stages, accuracies, marker='o', linewidth=2, markersize=8)
    plt.axhline(y=40, color='r', linestyle='--', label='Baseline (40%)')
    plt.axhline(y=60, color='g', linestyle='--', label='Target (60%)')
    plt.ylabel('Accuracy (%)')
    plt.xlabel('Training Stage')
    plt.title('Qwen3-4B Curriculum Learning Curve')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig('curriculum_curve_colab.png', dpi=150)
    plt.show()
    print("\nChart saved: curriculum_curve_colab.png")

## Step 6: Download Results

In [ ]:
from google.colab import files
import glob

# Download all result files
print("Downloading results...")
for stage in ["s1", "s2", "s3", "s4", "s5"]:
    result_file = f"eval_{stage}_colab.json"
    matching = glob.glob(result_file)
    if matching:
        files.download(matching[0])
        print(f"Downloaded: {result_file}")

# Download chart
if glob.glob("curriculum_curve_colab.png"):
    files.download("curriculum_curve_colab.png")
    print("Downloaded: curriculum_curve_colab.png")

---

## Notes

- **GPU Memory**: If OOM error occurs, reduce `--batch-size` in the evaluation script
- **Speed**: Expected ~1-2 samples/sec on T4 GPU (~5-8 minutes for 500 samples)
- **Results**: JSON files contain detailed predictions for error analysis